# PS 3 Final Exam Study Guide

## Jupyter notebook version (questions and R output)

This notebook contains the whole study guide: every question, with the R code that makes each figure placed right where the figure is used.

**To make the screenshots for the LaTeX version:** run the first code cell, then run each figure cell. Under each "Figure" heading below, take a screenshot of the code cell and its output and save it in a folder called `reference-images` using the file name written under the heading. The two plot figures (Figures 9 and 15) are saved automatically by their code cells, so they do not need a screenshot. The checklist is also in `SCREENSHOT_CHECKLIST.md`.

The answer key with explanations is in the LaTeX and Rmd versions of the study guide, not in this notebook.

In [ ]:
# Run this cell first
#install.packages("estimatr")
library(estimatr)
#install.packages("ggplot2")
library(ggplot2)
#install.packages("wooldridge")
library(wooldridge)
#install.packages("rdrobust")
library(rdrobust)
#install.packages("stargazer")
library(stargazer)
options(warn = -1)

# Folder where the plots get saved
dir.create("reference-images", showWarnings = FALSE)

## Real data, downloaded from the internet or loaded from R packages
social <- read.csv("https://raw.githubusercontent.com/kosukeimai/qss/master/CAUSALITY/social.csv")
resume <- read.csv("https://raw.githubusercontent.com/kosukeimai/qss/master/CAUSALITY/resume.csv")
minwage <- read.csv("https://raw.githubusercontent.com/kosukeimai/qss/master/CAUSALITY/minwage.csv")
data("rdrobust_RDsenate")
senate <- rdrobust_RDsenate
# vote1 comes with the wooldridge package

## Small data prep used by later cells
minwage$state <- ifelse(minwage$location == "PA", "PA", "NJ")
minwage$full_change <- minwage$fullAfter - minwage$fullBefore
senate$dem_won <- ifelse(senate$margin > 0, 1, 0)
voted.2004 <- subset(social, primary2004 == 1)
did.not.vote.2004 <- subset(social, primary2004 == 0)
close.races <- subset(senate, abs(margin) < 2)

## Part 1: Foundations (Weeks 2–6 review)

This part reviews the ideas from the first half of the course that everything else builds on. Questions in this part use the following data, which come from a real field experiment: Gerber, Green, and Larimer's (2008) social pressure experiment, in which registered voters were randomly assigned to receive different mailers before the August 2006 primary election in Michigan (data from Imai's *Quantitative Social Science* textbook). Each row is one voter.

- `sex`: `"female"` or `"male"`
- `yearofbirth`: the voter's year of birth
- `primary2004`: `1` if the voter voted in the August 2004 primary, `0` if not
- `messages`: the mailer condition the voter was randomly assigned to: `"Control"` (no mailer), `"Civic Duty"` (a mailer telling people it is their civic duty to vote), `"Hawthorne"` (a mailer saying "YOU ARE BEING STUDIED" and that turnout is being observed), or `"Neighbors"` (a mailer showing the voting records of the voter and their neighbors)
- `primary2006`: the **outcome**, `1` if the voter voted in the August 2006 primary, `0` if not
- `hhsize`: the number of people in the voter's household

### 1.1 Potential outcomes and causality

**Question 1.** Which of the following best describes a **potential outcome**?

- (a) The value of the outcome a particular unit would have had if it had received a particular treatment
- (b) The outcome we actually observe for a unit after treatment
- (c) The average outcome among everyone who received the treatment
- (d) The difference between the treatment group average and the control group average

For the next two questions, here are the (hypothetical) potential outcomes for four registered voters. The outcome is whether the voter turns out (`1` = votes, `0` = does not vote). The treatment is receiving a mailer urging them to vote.

| Voter | Turnout without the mailer | Turnout with the mailer |
|---|---|---|
| A | 0 | 1 |
| B | 1 | 1 |
| C | 0 | 0 |
| D | 0 | 1 |

**Question 2.** What is the average treatment effect of the mailer for these four voters?

- (a) 0.25
- (b) 0.50
- (c) 0.75
- (d) 1.00

**Question 3.** Why can't researchers compute these unit-level treatment effects in a real study?

- (a) Each voter can only receive one treatment, so we only ever see one of that voter's two potential outcomes
- (b) The sample is usually too small
- (c) Treatment effects are not defined for individual people
- (d) Potential outcomes only exist in randomized experiments

**Question 4.** Which of the following are **causal** questions? *Select all*

- (a) Does receiving a mailer showing your neighbors' voting records increase the probability that you vote?
- (b) Do women vote at higher rates than men?
- (c) Would this particular voter have voted if she had not received the mailer?
- (d) Which state had the highest turnout rate in 2020?

### 1.2 Correlation is not causation

**Question 5.** A study of US House incumbents finds that the incumbents who spend more money on their campaigns tend to win a *smaller* share of the vote. Which of the following are possible explanations that do **not** mean spending causes worse results? *Select all*

- (a) Reverse causation: incumbents who expect a close race raise and spend more money
- (b) Omitted variable bias: how competitive the district is could cause both higher spending and a smaller vote share
- (c) Because the correlation is negative, spending must have no effect on vote share
- (d) Collecting data on every House district in the country would remove the problem

**Question 6.** A university compares the final exam scores of students who chose to attend optional review sessions with the scores of students who did not, and finds that attendees scored 8 points higher. What is the best critique of the claim that the review sessions caused the higher scores?

- (a) Students who choose to attend may be different (e.g., more motivated) than students who do not, so the groups would differ even without the sessions (selection bias)
- (b) The number of students is too small to compare two averages
- (c) Exam scores cannot be compared across groups
- (d) There is no problem: a difference in averages always measures the causal effect

### 1.3 Randomized experiments

#### Figure 1

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig01-social-balance.png`.

In [ ]:
table(social$messages, social$sex)

**Question 7.** In Figure 1, how many women were assigned to the Hawthorne condition?

- (a) 19,064
- (b) 19,140
- (c) 38,204
- (d) 95,419

**Question 8.** Which of the following are true about Figure 1? *Select all*

- (a) The share of women is nearly identical across the four conditions (about 50\% in each), which is what we expect when assignment is random
- (b) Random assignment should also balance characteristics the researchers did not measure
- (c) The Control group is much larger than the others, so it must differ from the treatment groups in other ways
- (d) If the shares of women had differed by a few percentage points, that would prove assignment was not random

**Question 9.** Why is comparing the 2006 turnout of the Neighbors group to the 2006 turnout of the Control group free of selection bias?

- (a) Voters were randomly assigned to each group, so the two groups have the same potential outcomes on average
- (b) The sample is very large
- (c) The voters chose which mailer to receive
- (d) The researchers controlled for sex and age in the comparison

**Question 10.** Which of the following are true?<br>(1) In an experiment with random assignment, the treatment and control groups are, on average, identical on characteristics the researchers never measured.<br>(2) Experiments eliminate noise as well as bias.

- (a) (1) only
- (b) (2) only
- (c) Both (1) and (2)
- (d) Neither (1) nor (2)

### 1.4 Reading difference-in-means output

The next questions use another real experiment: Bertrand and Mullainathan's (2004) resume audit study. Researchers sent fictitious resumes in response to real job ads and randomly assigned each resume a first name that signals the applicant's race. Each row is one resume.

- `race`: `"black"` or `"white"`, the race signaled by the name randomly assigned to the resume
- `call`: the **outcome**, `1` if the employer called back, `0` if not
- `sex`, `firstname`: the sex and first name on the resume

#### Figure 2

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig02-resume-table.png`.

In [ ]:
table(resume$race, resume$call)

**Question 11.** In Figure 2, about what percent of the resumes with Black-sounding names received a callback?

- (a) 6.4\%
- (b) 9.6\%
- (c) 40.1\%
- (d) 93.6\%

#### Figure 3

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig03-resume-dim.png`.

In [ ]:
difference_in_means(call ~ race, resume, condition1 = 'black', condition2 = 'white')

**Question 12.** Which of the following is the correct interpretation of the estimate in Figure 3?

- (a) In this experiment, resumes with white-sounding names received callbacks 3.2 percentage points more often than otherwise similar resumes with Black-sounding names
- (b) Resumes with white-sounding names were 3.2\% more likely to get a callback (a relative increase of 3.2\%)
- (c) Resumes with Black-sounding names received callbacks 3.2 percentage points more often
- (d) Among all job applicants in the US, white applicants are called back 3.2 percentage points more often

**Question 13.** What would change in the output of Figure 3 if the code used `condition1 = 'white'` and `condition2 = 'black'` instead?

- (a) The estimate would be about $-0.032$ and the confidence interval would flip sign; the standard error and p-value would stay the same
- (b) The standard error would change sign
- (c) The p-value would become about 0.96
- (d) Nothing would change at all

### 1.5 Bias, noise, and the standard error

**Question 14.** In 1936 the *Literary Digest* surveyed 2.2 million people and predicted that Landon would beat Roosevelt in a landslide; Roosevelt won easily. Which of the following are true? *Select all*

- (a) The huge sample made the noise in the poll very small, but it did nothing to fix the bias from who was surveyed
- (b) Gallup's much smaller but more representative sample got closer to the truth
- (c) The poll missed because 2.2 million respondents is too few to measure a national election
- (d) The error was due to noise, so repeating the poll would likely have given the right answer

**Question 15.** A researcher wants to cut the standard error of a difference-in-means estimate in half. If nothing else changes (including the spread of the outcome and the split between groups), the sample size needs to become...

- (a) Twice as large
- (b) Four times as large
- (c) Half as large
- (d) Ten times as large

**Question 16.** A study's estimate is $0.12$ and its standard error is $0.06$. What is the t-statistic, and is the estimate statistically significant by the usual convention?

- (a) $t = 2$; statistically significant, because $|t| > 1.96$
- (b) $t = 0.5$; not statistically significant
- (c) $t = 0.0072$; not statistically significant
- (d) $t = 2$; not statistically significant, because the p-value is the same as the t-statistic

**Question 17.** A researcher has 10,000 people to divide between a treatment group and a control group. Which split gives the smallest standard error (if the outcome has the same spread in both groups)?

- (a) 5,000 treatment and 5,000 control
- (b) 9,000 treatment and 1,000 control
- (c) 1,000 treatment and 9,000 control
- (d) All splits give the same standard error

## Part 2: p-values and confidence intervals (Week 7)

These questions revisit the experiments above. Figure 3 is the resume experiment. Figure 4 compares the Hawthorne mailer to the Civic Duty mailer, with Civic Duty as the baseline:

#### Figure 4

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig04-hawthorne-vs-civic.png`.

In [ ]:
difference_in_means(primary2006 ~ messages, social, condition1 = 'Civic Duty', condition2 = 'Hawthorne')

**Question 18.** What is the **null hypothesis** a skeptic would hold about the resume experiment in Figure 3?

- (a) The race signaled by the name has no effect on the probability of a callback
- (b) The race signaled by the name increases the probability of a callback
- (c) The estimate equals exactly 0.032
- (d) The experiment was not randomized

**Question 19.** Which of the following is the correct interpretation of the p-value (about $0.00004$) in Figure 3?

- (a) If the race signaled by the name had no effect, there would be only about a 0.004\% chance of seeing an estimate this large or larger
- (b) There is a 0.004\% chance that the race signaled by the name has no effect
- (c) There is a 99.996\% chance that the result is real
- (d) The probability that the estimate is due to chance is 0.004\%

**Question 20.** Which of the following are true about Figure 3? *Select all*

- (a) The t value is the estimate divided by the standard error: $0.0320/0.00778 \approx 4.1$
- (b) The estimate is statistically significant, because the p-value is below 0.05 (and $|t| > 1.96$)
- (c) Statistical significance shows that the effect of the name is large
- (d) The p-value proves that the null hypothesis is false with certainty

**Question 21.** Which of the following is the best interpretation of the 95\% confidence interval ($0.017$ to $0.047$) in Figure 3?

- (a) The most pessimistic plausible effect is about 1.7 percentage points and the most optimistic plausible effect is about 4.7 percentage points
- (b) 95\% of employers called back white-named resumes between 1.7\% and 4.7\% more often than Black-named resumes
- (c) The effect is exactly 3.2 percentage points
- (d) There is a 5\% chance that the experiment was done incorrectly

**Question 22.** Using the estimate and standard error in Figure 4 (estimate $0.0078$, standard error $0.0034$), what is the 95\% confidence interval, rounded to four decimals?

- (a) $0.0011$ to $0.0145$
- (b) $0.0044$ to $0.0112$
- (c) $0.0078$ to $0.0145$
- (d) $-0.0056$ to $0.0212$

**Question 23.** Which of the following are true?<br>(1) By the usual convention, we reject the null that the Hawthorne and Civic Duty mailers are equally effective, because the p-value in Figure 4 (about 0.02) is below 0.05.<br>(2) A p-value of 0.02 means there is a 2\% chance that the null hypothesis is true.

- (a) (1) only
- (b) (2) only
- (c) Both (1) and (2)
- (d) Neither (1) nor (2)

**Question 24.** A campaign is deciding whether to send the Hawthorne mailer instead of the Civic Duty mailer to 100,000 voters. Using the 95\% confidence interval in Figure 4, about how many *additional* voters would vote in the most pessimistic and most optimistic plausible scenarios?

- (a) Between about 120 and 1,440 additional voters
- (b) Exactly 784 additional voters
- (c) Between 0 and 7,840 additional voters
- (d) Between 12 and 144 additional voters

#### Figure 5

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig05-neighbors-vs-control.png`.

In [ ]:
difference_in_means(primary2006 ~ messages, social, condition1 = 'Control', condition2 = 'Neighbors')

**Question 25.** Figure 5 reports a p-value of $9.6 \times 10^{-199}$ for the Neighbors mailer relative to Control. Which of the following are true? *Select all*

- (a) The estimate (about 8.1 percentage points) tells us the size of the effect; the p-value does not
- (b) The p-value is so small partly because the huge sample gives a very small standard error
- (c) A p-value this small means the Neighbors mailer had an enormous effect
- (d) The p-value means that there is a $9.6 \times 10^{-199}$ probability that the null hypothesis is true

**Question 26.** Experiment A finds an effect of 8 percentage points with a standard error of 30 percentage points. Experiment B finds an effect of 5 percentage points with a standard error of 1 percentage point. Which conclusion is best supported?

- (a) Experiment B is much more informative: Experiment A's interval is so wide that it is compatible with large harms or large benefits
- (b) We should be more optimistic about A, because its upper bound is much higher
- (c) The two experiments provide equally strong evidence, because both estimates are positive
- (d) Experiment A shows the treatment works better, because its estimate is larger

**Question 27.** An experiment finds an estimate of +3 percentage points with a p-value of 0.30. Which of the following are true? *Select all*

- (a) We cannot rule out that a skeptic who believes the treatment does nothing is right
- (b) The +3 points is still our best guess of the treatment effect
- (c) The experiment proves the treatment has no effect
- (d) There is a 30\% chance that the treatment has no effect

## Part 3: Heterogeneous effects and generalizability (Week 8)

The next figures estimate the effect of the Neighbors mailer relative to Control separately for voters who voted in the 2004 primary (Figure 6) and voters who did not vote in the 2004 primary (Figure 7).

#### Figure 6

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig06-hte-voted-2004.png`.

In [ ]:
difference_in_means(primary2006 ~ messages, voted.2004, condition1 = 'Control', condition2 = 'Neighbors')

#### Figure 7

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig07-hte-did-not-vote-2004.png`.

In [ ]:
difference_in_means(primary2006 ~ messages, did.not.vote.2004, condition1 = 'Control', condition2 = 'Neighbors')

**Question 28.** Which code produced Figure 6?

- (a) `voted.2004 <- subset(social, primary2004 == 1)`<br>`difference_in_means(primary2006 ~ messages, voted.2004, condition1 = 'Control', condition2 = 'Neighbors')`
- (b) `did.not.vote.2004 <- subset(social, primary2004 == 0)`<br>`difference_in_means(primary2006 ~ messages, did.not.vote.2004, condition1 = 'Control', condition2 = 'Neighbors')`
- (c) `difference_in_means(primary2006 ~ primary2004, social, condition1 = 0, condition2 = 1)`
- (d) `neighbors <- subset(social, messages == 'Neighbors')`<br>`difference_in_means(primary2006 ~ primary2004, neighbors, condition1 = 0, condition2 = 1)`

**Question 29.** Based on Figure 6 and Figure 7, which is our best guess about the effect of the Neighbors mailer?

- (a) It was larger among voters who voted in the 2004 primary (about 9.7 points) than among those who did not (about 6.9 points)
- (b) It was larger among voters who did not vote in the 2004 primary
- (c) It was about the same in both groups, because both are statistically significant
- (d) It cannot be estimated for either group, because voting in 2004 was not randomly assigned

**Question 30.** Which of the following are true?<br>(1) Within each subgroup, the estimated effect of the mailer is causal, because the mailer was randomly assigned.<br>(2) The comparison of the effect between 2004 voters and non-voters shows that voting in 2004 *causes* people to respond more strongly to mailers.

- (a) (1) only
- (b) (2) only
- (c) Both (1) and (2)
- (d) Neither (1) nor (2)

**Question 31.** A researcher tests whether a treatment works in 40 different subgroups of the data and finds 2 statistically significant effects. Suppose the treatment actually has no effect for anyone. About how many significant subgroup results (at $p < 0.05$) would we expect from chance alone, and what is this problem called?

- (a) About 2; p-hacking
- (b) About 0; omitted variable bias
- (c) About 20; selection bias
- (d) About 40; reverse causation

**Question 32.** Which of the following would help reduce the risk of p-hacking when looking for heterogeneous effects? *Select all*

- (a) Choosing the subgroups to examine before running the experiment
- (b) Being more skeptical of a p-value just under 0.05 when many subgroups have been tested
- (c) Testing as many subgroups as possible and reporting only the significant ones
- (d) Using a bigger sample, since that guarantees no false positives

**Question 33.** Figure 5 estimates the effect of the Neighbors mailer in an August 2006 primary in Michigan. Which of the following are reasons the effect might be different in a 2024 presidential general election in a swing state? *Select all*

- (a) Voters in a swing state receive huge amounts of other campaign mail, so one more mailer may matter less
- (b) Turnout is already much higher in a presidential general election, leaving less room for a mailer to raise it
- (c) The people who vote in a presidential election are different from the people who vote in a low-turnout primary
- (d) Because the original experiment was randomized, the result automatically generalizes to any election

**Question 34.** A meta-analysis looks at two experiments. Experiment 1 estimates an effect of 8 percentage points with a standard error of 30. Experiment 2 estimates 5 percentage points with a standard error of 1. About what is the precision-weighted average effect?

- (a) 6.5
- (b) 5.0
- (c) 8.0
- (d) 13

**Question 35.** Which of the following are true?<br>(1) Conducting a meta-analysis aggregates the findings of previous research.<br>(2) Conducting a meta-analysis eliminates the biases from flawed research designs.

- (a) (1) only
- (b) (2) only
- (c) Both (1) and (2)
- (d) Neither (1) nor (2)

## Part 4: Describing populations (Week 9)

**Question 36.** Which of the following are true about the *Literary Digest* poll, which surveyed 2.2 million people in 1936 and wrongly predicted that Landon would win? *Select all*

- (a) Its sample was biased toward wealthier Americans (for example, magazine subscribers), and a larger sample does not fix that kind of bias
- (b) A sample can be huge and still produce an answer that is far from the truth
- (c) The poll was wrong because 2.2 million people is too small a sample
- (d) A random sample of just 100 people would have been less accurate than the Digest's poll

**Question 37.** A pollster draws a random sample of 1,000 people from a voter file and calls each of them, but only 5\% pick up, and the people who pick up tend to be older than the people who do not. Which problem does this describe?

- (a) Non-response bias
- (b) Sample selection bias
- (c) Reverse causation
- (d) There is no bias, because the original sample was random

**Question 38.** A mayor asks everyone who comes to a town hall meeting whether they support a new bus lane and concludes that residents oppose it. Which problem is the biggest threat to this conclusion?

- (a) Sample selection bias: people who choose to attend a town hall differ from residents overall
- (b) Omitted variable bias in a regression
- (c) Noise, because town halls are small
- (d) Reverse causation: opposing the bus lane causes people to attend

**Question 39.** A survey sample is 75\% male and 25\% not male, while the population is 50\% male and 50\% not male. Relative to each male respondent, about how much should each non-male respondent count in a weighted analysis?

- (a) $1/3$ as much
- (b) The same amount
- (c) 3 times as much
- (d) 25 times as much

**Question 40.** Three survey respondents answered a yes/no question ($1$ = yes). Respondent 1 answered $1$ and has weight 1. Respondent 2 answered $0$ and has weight 1. Respondent 3 answered $0$ and has weight 2. What is the weighted mean?

- (a) 0.33
- (b) 0.25
- (c) 0.50
- (d) 1.00

**Question 41.** Which of the following are true?<br>(1) Weights count under-represented groups more so that the sample looks more like the population on the variables we weight on.<br>(2) After weighting, a sample is guaranteed to be representative on every variable, even ones that were not used for weighting.

- (a) (1) only
- (b) (2) only
- (c) Both (1) and (2)
- (d) Neither (1) nor (2)

The next question asks a *descriptive* question with the social pressure data: did men and women differ in whether they voted in the 2004 primary? Because the baseline is `'female'` and the comparison group is `'male'`, the estimate is the male average minus the female average.

#### Figure 8

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig08-sex-difference-2004.png`.

In [ ]:
difference_in_means(primary2004 ~ sex, social, condition1 = 'female', condition2 = 'male')

**Question 42.** Which of the following is the correct interpretation of the estimate in Figure 8?

- (a) In this sample, men were about 0.16 percentage points more likely than women to have voted in the 2004 primary
- (b) Men are 0.16\% more likely to vote than women in the US
- (c) Women were 0.16 percentage points more likely to have voted in the 2004 primary
- (d) Being male causes people to vote at higher rates

**Question 43.** Which of the following are true about Figure 8? *Select all*

- (a) The difference is not statistically significant, because the p-value (about 0.37) is above 0.05
- (b) The 95\% confidence interval (about $-0.19$ to $0.51$ percentage points) includes 0
- (c) This shows that men and women voted at exactly the same rates in the population
- (d) The p-value means there is a 37\% chance that men and women do not differ

## Part 5: Scatterplots and regression (Weeks 10–11)

The next questions use real data on 173 US House elections in 1988 (the `vote1` dataset in the `wooldridge` R package). Each row is one election with two candidates, called candidate A and candidate B.

- `voteA`: the percent of the vote candidate A received
- `expendA`, `expendB`: how much candidate A and candidate B spent on their campaigns, in *thousands of dollars*
- `prtystrA`: the percent of the vote that candidate A's party's presidential candidate received in the district in the previous presidential election (a measure of how strong A's party is in the district)

#### Figure 9

This plot is saved automatically as `reference-images/final-fig09-spending-scatter.png` when you run the cell. (No screenshot needed.)

In [ ]:
spending.plot <- qplot(expendA, voteA, data = vote1) + geom_smooth(method = 'lm', formula = y ~ x)
ggsave('reference-images/final-fig09-spending-scatter.png', spending.plot, width = 6, height = 4, dpi = 200)
spending.plot

**Question 44.** Which pair of commands correctly makes a scatterplot with `expendA` on the x-axis and `voteA` on the y-axis, and then runs a regression predicting `voteA` from `expendA`?

- (a) `qplot(expendA, voteA, data = vote1)` and `lm(voteA ~ expendA, data = vote1)`
- (b) `qplot(voteA, expendA, data = vote1)` and `lm(voteA ~ expendA, data = vote1)`
- (c) `qplot(expendA, voteA, data = vote1)` and `lm(expendA ~ voteA, data = vote1)`
- (d) `qplot(voteA, expendA, data = vote1)` and `lm(expendA ~ voteA, data = vote1)`

**Question 45.** Which of the following is true about the line in Figure 9?

- (a) It is our best guess of `voteA` for each value of `expendA`
- (b) It connects the highest and lowest points in the data
- (c) It shows that spending causes candidates to win more of the vote
- (d) It shows the exact vote share for every candidate

**Question 46.** In Figure 9, a straight line is not a perfect summary of the pattern. Which of the following is a pattern in the points that the line does **not** capture?

- (a) Many candidates who spend little have low vote shares, many who spend a moderate amount have high vote shares, and the few candidates who spend over 1,000 have middling vote shares
- (b) Every point falls exactly on the line
- (c) Candidates who spend more always receive a lower vote share
- (d) The data contain only two possible values of `voteA`

**Question 47.** Because `expendA` is measured in thousands of dollars, a prediction for a candidate who spends \$5 million (`expendA = 5000`) would be unreliable. Why?

- (a) The highest spending in the data is about 1,470 thousand dollars, so this would be extrapolating far beyond the data
- (b) Regression cannot make predictions for any value of `expendA`
- (c) The predicted vote share would be negative
- (d) The line of best fit changes whenever we ask for a new prediction

#### Figure 10

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig10-lm-bivariate.png`.

In [ ]:
summary(lm(voteA ~ expendA, data = vote1))

**Question 48.** Using Figure 10, which is the best interpretation of the coefficient on `expendA` (about 0.0236)?

- (a) Candidates who spent \$1,000 more received, on average, about 0.024 percentage points more of the vote in these data
- (b) Spending an additional \$1,000 *causes* about 0.024 percentage points more of the vote
- (c) Candidates who spent \$1 more received 0.0236 more percentage points of the vote
- (d) The average candidate in these data received 0.0236\% of the vote

**Question 49.** Using Figure 10, what vote share would the regression predict for a candidate who spent \$500,000 (`expendA = 500`)?

- (a) 43.2
- (b) 55.0
- (c) 66.0
- (d) 11.8

**Question 50.** In Figure 10, how should we interpret the `(Intercept)` of about 43.2?

- (a) The regression's best guess of `voteA` for a candidate who spent \$0
- (b) The average amount spent by candidates
- (c) The effect of spending on vote share
- (d) The standard error of the estimate

**Question 51.** Which of the following are reasons why the relationship in Figure 10 may **not** be causal? *Select all*

- (a) Omitted variable bias: for example, the strength of a candidate's party in the district could affect both spending and vote share
- (b) Reverse causation: candidates who expect to do well may find it easier to raise money
- (c) The sample is too small to run a regression
- (d) The relationship is statistically significant

**Question 52.** A researcher adds random noise to the outcome variable and re-runs a bivariate regression. Which of the following are true?<br>(1) The slope of the line of best fit will be about the same.<br>(2) The correlation between the variables will be about the same.

- (a) (1) only
- (b) (2) only
- (c) Both (1) and (2)
- (d) Neither (1) nor (2)

The next figure adds the other candidate's spending (`expendB`) and party strength to the regression.

#### Figure 11

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig11-lm-multivariate.png`.

In [ ]:
summary(lm(voteA ~ expendA + expendB + prtystrA, data = vote1))

**Question 53.** In Figure 11, what is the best interpretation of the coefficient on `expendA` (about 0.0349)?

- (a) Among races where candidate B's spending and A's party strength are the same, candidates who spent \$1,000 more received about 0.035 percentage points more of the vote on average
- (b) Spending \$1,000 more causes candidate A to receive 0.035 percentage points more of the vote
- (c) Candidates who spent \$1,000 more received 0.035 percentage points more of the vote, regardless of the other variables
- (d) It is the average of candidate A's spending

**Question 54.** The coefficient on `expendA` is about 0.0236 in Figure 10 but about 0.0349 in Figure 11. Which is the best explanation?

- (a) The bivariate regression left out variables (like the opponent's spending) that are related to both `expendA` and `voteA`, so the estimates differ once they are included
- (b) The second regression is just a different sample of the same elections
- (c) Adding more variables to a regression always gives the correct causal estimate
- (d) The difference is only due to rounding

**Question 55.** Using Figure 11, what is the best interpretation of the coefficient on `expendB` (about $-0.0349$)?

- (a) Holding candidate A's spending and party strength constant, races where candidate B spent \$1,000 more had about 0.035 percentage points less of the vote for candidate A, on average
- (b) Candidate B's spending has no statistically significant relationship with `voteA`
- (c) Candidate B spending \$1,000 more causes candidate A to lose exactly 0.035 percentage points
- (d) A's vote share goes up by 0.035 for every \$1,000 candidate B spends

**Question 56.** Using Figure 11, what vote share would the regression predict for a candidate when both candidates spent \$300,000 (`expendA = 300` and `expendB = 300`) and `prtystrA = 50`?

- (a) About 50.4
- (b) About 33.3
- (c) About 60.9
- (d) About 17.1

**Question 57.** Which of the following are true about multivariate regression with observational data?<br>(1) Controlling for additional variables can reduce omitted variable bias.<br>(2) Once we control for a few important variables, we can be sure that no omitted variable bias remains.

- (a) (1) only
- (b) (2) only
- (c) Both (1) and (2)
- (d) Neither (1) nor (2)

The next questions analyze the social pressure *experiment* with a regression instead of `difference_in_means()`.

#### Figure 12

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig12-lm-experiment.png`.

In [ ]:
summary(lm(primary2006 ~ messages, data = social))

**Question 58.** In Figure 12, which condition is the baseline (the omitted category) that all of the coefficients are compared to?

- (a) Control
- (b) Civic Duty
- (c) Hawthorne
- (d) Neighbors

**Question 59.** What is the best interpretation of the `(Intercept)` of about 0.315 in Figure 12?

- (a) The average 2006 turnout in the baseline group (voters who received the Civic Duty mailer) was about 31.5\%
- (b) The average 2006 turnout in the Control group was about 31.5\%
- (c) The effect of the Civic Duty mailer relative to no mailer was 31.5 percentage points
- (d) Overall turnout in the sample was 31.5\%

**Question 60.** Using Figure 12, what is the estimated effect of the Neighbors mailer relative to the *Control* group?

- (a) 0.063
- (b) About 0.081
- (c) About $-0.018$
- (d) 0.315

**Question 61.** Using the estimate and standard error for `messagesHawthorne` in Figure 12 (estimate 0.0078, standard error 0.0033), what is the 95\% confidence interval, rounded to three decimals?

- (a) $0.001$ to $0.014$
- (b) $0.005$ to $0.011$
- (c) $0.000$ to $0.008$
- (d) $-0.013$ to $0.029$

## Part 6: Reading regression tables (Week 12)

#### Figure 13

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig13-table-observational.png`.

In [ ]:
model.1 <- lm(voteA ~ expendA, data = vote1)
model.2 <- lm(voteA ~ expendA + prtystrA, data = vote1)
model.3 <- lm(voteA ~ expendA + expendB + prtystrA, data = vote1)
stargazer(model.1, model.2, model.3, type = 'text')

**Question 62.** How many separate regressions are reported in Figure 13?

- (a) 1
- (b) 2
- (c) 3
- (d) 4

**Question 63.** Which of the following is true about the blank cells in the `expendB` row of Figure 13?

- (a) `expendB` was not included in regressions (1) and (2)
- (b) `expendB` had a coefficient of exactly zero in regressions (1) and (2)
- (c) `expendB` was not statistically significant in regressions (1) and (2)
- (d) There is an error in the table

**Question 64.** In column (3) of Figure 13, the `expendA` row reads `0.035***` with `(0.003)` below it. Which of the following are true? *Select all*

- (a) The number in parentheses is the standard error
- (b) The coefficient is statistically significant at the 0.01 level (the t-statistic is roughly 0.035/0.003, which is much bigger than 1.96)
- (c) The number in parentheses is the p-value
- (d) 0.003 is the average amount candidate A spent

**Question 65.** What does 173 next to "Observations" mean in Figure 13?

- (a) The regressions use 173 House elections, one row of data per election
- (b) 173 candidates spent more than average
- (c) The coefficient on `expendA` is 173
- (d) 173 variables are included in each regression

**Question 66.** What is the difference between column (1) and column (2) in Figure 13, and what happens to the `expendA` coefficient?

- (a) Column (2) adds `prtystrA` as a control variable; the coefficient on `expendA` falls from 0.024 to 0.020
- (b) Column (2) uses a different outcome variable; the coefficient on `expendA` stays the same
- (c) Column (2) drops `expendA`; the coefficient falls to zero
- (d) Column (2) uses a different set of elections; the coefficient on `expendA` rises

**Question 67.** Which of the following are true?<br>(1) The $R^2$ in column (3) of Figure 13 (0.569) is higher than in column (1) (0.156), so the variables in column (3) explain more of the variation in `voteA`.<br>(2) A higher $R^2$ shows that the variables in the regression cause the outcome.

- (a) (1) only
- (b) (2) only
- (c) Both (1) and (2)
- (d) Neither (1) nor (2)

#### Figure 14

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig14-table-experimental.png`.

In [ ]:
experiment.1 <- lm(primary2006 ~ messages, data = social)
experiment.2 <- lm(primary2006 ~ messages + sex + yearofbirth + primary2004 + hhsize, data = social)
stargazer(experiment.1, experiment.2, type = 'text')

**Question 68.** Which coefficients in column (2) of Figure 14 can be interpreted as causal effects?

- (a) The `messages` coefficients, because the mailers were randomly assigned
- (b) All of the coefficients, because the data come from an experiment
- (c) Only `primary2004`, since it has the largest coefficient
- (d) None of them, since regressions can never show causal effects

**Question 69.** Why might researchers report column (2) of Figure 14, which adds control variables to the experiment?<br>(1) Including control variables can increase the precision of the treatment effect estimates.<br>(2) Including control variables is necessary for the estimates to be unbiased, because otherwise the treatment and control groups might differ.

- (a) (1) only
- (b) (2) only
- (c) Both (1) and (2)
- (d) Neither (1) nor (2)

**Question 70.** In column (2) of Figure 14, the coefficient on `yearofbirth` is $-0.004$. Which is the best interpretation?

- (a) Among voters with the same mailer, sex, 2004 voting history, and household size, voters born one year later were about 0.4 percentage points less likely to vote in 2006 on average
- (b) Being born one year later causes a voter to be 0.4 percentage points less likely to vote
- (c) Voters were randomly assigned a birth year
- (d) Voters born later were 0.004 percentage points less likely to vote

**Question 71.** Using the `messagesNeighbors` row of column (1) of Figure 14 (0.063 with a standard error of 0.003), what is the approximate 95\% confidence interval?

- (a) About 0.057 to 0.069
- (b) About 0.060 to 0.066
- (c) About 0.063 to 0.069
- (d) About 0.000 to 0.126

## Part 7: Natural experiments, instrumental variables, and regression discontinuity (Week 13)

In the 1960s and 1970s, the US military drafted young men for the Vietnam War using a **lottery**: each birthday was randomly assigned a number, and men with low numbers were much more likely to be drafted. Angrist (1990) used this to study the effect of serving in the military on later earnings.

**Question 72.** In this design, which is the **instrument**, the **treatment variable**, and the **outcome**?

- (a) Instrument: draft lottery number<br>Treatment: military service<br>Outcome: later earnings
- (b) Instrument: military service<br>Treatment: draft lottery number<br>Outcome: later earnings
- (c) Instrument: later earnings<br>Treatment: military service<br>Outcome: draft lottery number
- (d) Instrument: military service<br>Treatment: later earnings<br>Outcome: draft lottery number

**Question 73.** Which of the following would be a violation of the **exclusion restriction** in this study? *Select all*

- (a) Men with low lottery numbers were more likely to enroll in college to get a deferment, and college raises earnings
- (b) Men with low lottery numbers were more likely to move to Canada, which affects their earnings
- (c) Men with low lottery numbers were more likely to be drafted into military service
- (d) Lottery numbers were randomly assigned by birthday

**Question 74.** In an instrumental variables study, what does the **first stage** tell us?

- (a) How strongly the instrument changes the treatment variable (for example, how much winning a school lottery raises the years a student spends at the school)
- (b) Whether the treatment variable changes the outcome
- (c) Whether the exclusion restriction is true
- (d) Whether the instrument is randomly assigned

**Question 75.** "In order to determine whether $X$ causes $Y$, you must understand what causes $X$." Which question does this advice tell us to ask?

- (a) Who am I comparing to whom, and how did they get the values of $X$ that they did?
- (b) What are all of the other variables that cause $Y$?
- (c) How large is the sample?
- (d) How many control variables did the study include?

**Question 76.** Which statement about instrumental variables estimates compared to ordinary regression estimates is most accurate?

- (a) IV estimates usually rely on weaker assumptions than regression on observational data, but they usually have larger standard errors
- (b) IV estimates always have smaller standard errors than regression estimates
- (c) IV estimates are as credible as a randomized experiment in every case
- (d) IV estimates do not require any assumptions

Now a regression discontinuity design. Lee (2008) studied US Senate elections: does narrowly winning an election give a party an advantage in the next election for that seat? The data (the `rdrobust_RDsenate` dataset in the `rdrobust` R package) have one row per Senate race.

- `margin`: the Democratic candidate's margin of victory, in percentage points (positive = Democrat won, negative = Democrat lost)
- `vote`: the Democratic share of the vote in the *next* election for that seat
- `dem_won`: `1` if `margin` is positive, `0` if not (created in the code)

#### Figure 15

This plot is saved automatically as `reference-images/final-fig15-rdd-plot.png` when you run the cell. (No screenshot needed.)

In [ ]:
rdd.plot <- qplot(margin, vote, data = subset(senate, abs(margin) < 25)) + geom_smooth(aes(group = dem_won), method = 'lm', formula = y ~ x) + geom_vline(xintercept = 0, linetype = 'dashed')
ggsave('reference-images/final-fig15-rdd-plot.png', rdd.plot, width = 6, height = 4, dpi = 200)
rdd.plot

**Question 77.** In the regression discontinuity design behind Figure 15, what is the running variable, the cutoff, and the treatment?

- (a) Running variable: `margin`<br>Cutoff: 0<br>Treatment: the Democrat winning the election (`dem_won`)
- (b) Running variable: `vote`<br>Cutoff: 50<br>Treatment: `margin`
- (c) Running variable: `dem_won`<br>Cutoff: 0<br>Treatment: `vote`
- (d) Running variable: `margin`<br>Cutoff: 0<br>Treatment: `vote`

**Question 78.** Why do regression discontinuity studies compare candidates who *narrowly* won to candidates who *narrowly* lost, instead of comparing all winners to all losers?

- (a) Races decided by a tiny margin are essentially a coin flip, so narrow winners and narrow losers are similar except for who won, while landslide winners differ in many ways
- (b) Narrow races have more voters, which makes the estimate more precise
- (c) Landslide races are not included in the data
- (d) Narrow winners always have the same vote share as narrow losers

#### Figure 16

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig16-rdd-dim.png`.

In [ ]:
difference_in_means(vote ~ dem_won, close.races, condition1 = 0, condition2 = 1)

**Question 79.** Figure 16 estimates the effect among races decided by less than 2 percentage points. Which is the best interpretation of the estimate (about 10.2)?

- (a) Among close races, Democratic candidates who narrowly won received about 10 percentage points more of the vote in the next election than Democratic candidates who narrowly lost
- (b) Democratic candidates who won by any margin received 10.2 more percentage points in the next election
- (c) The Democratic candidates who narrowly lost received 10.2 more percentage points
- (d) Winning an election increases the vote share in the next election by 10.2\% in every race, including landslides

**Question 80.** Which of the following would threaten the validity of this regression discontinuity design? *Select all*

- (a) A party could influence the vote count when a race is very close, so that narrow winners are not like narrow losers
- (b) Candidates could precisely control their final vote margin to land just above 0
- (c) Races with large margins have different vote shares in the next election than close races
- (d) More Democrats won than lost in the data (750 versus 640 races)

**Question 81.** Which of the following are checks of an RDD's validity? *Select all*

- (a) Checking that other variables (such as the previous election's vote share) look similar just above and just below the cutoff
- (b) Checking that there is no unusual bunching of observations on one side of the cutoff
- (c) Checking that the effect is the same for races decided by 1 point and by 50 points
- (d) Making the sample bigger by including every race, whatever its margin

**Question 82.** A researcher re-runs Figure 16 using a much *narrower* window (races decided by less than 0.5 points instead of less than 2). What is the most likely consequence?

- (a) The races compared are more similar, but there are fewer of them, so the standard error will probably be larger
- (b) The standard error will get smaller, because narrow races are more precise
- (c) The estimate must be exactly the same
- (d) The estimate becomes causal only with the narrower window

**Question 83.** Which of the following is the main limitation on **generalizing** from this study?

- (a) The estimate applies to close elections near the cutoff, which may differ from other elections
- (b) The estimate applies only to the largest landslide races
- (c) The estimate cannot be generalized to any Senate race
- (d) Randomization was not used, so there is no limitation

## Part 8: Difference-in-differences and theories (Week 14)

In April 1992, New Jersey raised its minimum wage from \$4.25 to \$5.05 an hour, while neighboring Pennsylvania did not change its minimum wage. Card and Krueger (1994) surveyed fast-food restaurants in both states before and after the increase. The data come from Imai's *Quantitative Social Science* textbook, with one row per restaurant (358 restaurants).

- `state`: `"NJ"` or `"PA"` (created in the code from `location`)
- `fullBefore`, `fullAfter`: the number of full-time employees at the restaurant before and after the New Jersey minimum wage increase
- `full_change`: `fullAfter` minus `fullBefore` (created in the code)

#### Figure 17

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig17-did-means.png`.

In [ ]:
aggregate(cbind(fullBefore, fullAfter) ~ state, data = minwage, FUN = mean)

**Question 84.** In this study, which group is the treated group and which is the comparison (control) group?

- (a) New Jersey restaurants are treated; Pennsylvania restaurants are the comparison group
- (b) Pennsylvania restaurants are treated; New Jersey restaurants are the comparison group
- (c) Restaurants measured before the increase are treated; restaurants measured after are the comparison group
- (d) There is no comparison group in a difference-in-differences design

**Question 85.** Using Figure 17, what was the change in the average number of full-time employees per restaurant in New Jersey, and in Pennsylvania?

- (a) New Jersey: about +0.43; Pennsylvania: about $-2.49$
- (b) New Jersey: about $-2.49$; Pennsylvania: about +0.43
- (c) New Jersey: about 8.41; Pennsylvania: about 8.17
- (d) New Jersey: about +2.93; Pennsylvania: about 0

**Question 86.** Using Figure 17, what is the difference-in-differences estimate of the effect of the New Jersey minimum wage increase on full-time employment?

- (a) +0.43
- (b) $-2.49$
- (c) +2.93
- (d) +0.23

**Question 87.** Why not simply compare New Jersey and Pennsylvania restaurants *after* the increase, or look only at New Jersey before and after? *Select all*

- (a) Pennsylvania restaurants already had more full-time employees than New Jersey restaurants *before* the increase (about 10.7 versus 8.0), so an after-only comparison is affected by pre-existing differences
- (b) Employment changed in Pennsylvania too (it fell), so looking only at New Jersey over time would blame the minimum wage for changes that happened anyway
- (c) Difference-in-differences requires that restaurants be randomly assigned to states
- (d) A difference-in-differences design eliminates all bias, so the other comparisons are unnecessary

**Question 88.** What does the **parallel trends** assumption mean in this study?

- (a) If New Jersey had not raised its minimum wage, full-time employment would have changed by about the same amount in New Jersey as it did in Pennsylvania
- (b) New Jersey and Pennsylvania had the same number of full-time employees before the increase
- (c) Employment stayed constant in both states
- (d) The minimum wage increase had no effect on employment

**Question 89.** Which of the following would be a violation of the parallel trends assumption? *Select all*

- (a) A recession hit New Jersey's fast-food industry much harder than Pennsylvania's during 1992
- (b) Fast-food employment was already growing faster in New Jersey than in Pennsylvania in the years before 1992
- (c) Pennsylvania had more full-time employees than New Jersey before the increase
- (d) Both states had their employment affected by the same national economic trend

#### Figure 18

**Screenshot:** capture the code cell and its output below and save it as `reference-images/final-fig18-did-dim.png`.

In [ ]:
difference_in_means(full_change ~ state, minwage, condition1 = 'PA', condition2 = 'NJ')

**Question 90.** Which of the following are true about Figure 18? *Select all*

- (a) The estimate (about 2.9) matches the difference-in-differences calculation from the table of averages, because the outcome is each restaurant's *change* in full-time employees
- (b) The estimate is not statistically significant at the 0.05 level, because the p-value (about 0.09) is above 0.05 and the confidence interval includes 0
- (c) The 95\% confidence interval shows that the minimum wage increase definitely did not reduce employment
- (d) The p-value of 0.09 means there is a 9\% chance that the minimum wage increase had no effect

**Question 91.** Which of the following is an example of a **placebo test** for a difference-in-differences design? *Select all*

- (a) Checking whether employment in the two states was already moving in a similar way in several periods *before* the minimum wage increase
- (b) Checking for an "effect" of the New Jersey minimum wage increase on an outcome it should not affect, and hoping to find none
- (c) Checking whether the estimate gets larger when the sample gets larger
- (d) Checking whether restaurants in both states were randomly assigned to their state

**Question 92.** Why can't researchers use a difference-in-differences design in every study?

- (a) They need data from multiple time periods, including before the treatment, and a treatment that changes over time
- (b) The method only works with experimental data
- (c) The method requires a sample of at least a million observations
- (d) It cannot be used with outcomes that are numbers

The last questions are about theories and collective action. Two prisoners are interrogated separately. The table shows each prisoner's payoff, in years of prison expressed as negative numbers, so closer to zero is better (Prisoner P's payoff first, then Prisoner Q's).

| | Q confesses | Q does not confess |
|---|---|---|
| **P confesses** | $-4, -4$ | $-1, -8$ |
| **P does not confess** | $-8, -1$ | $-2, -2$ |

**Question 93.** Which of the following are true about this game? *Select all*

- (a) Whatever Q does, P gets a better payoff by confessing
- (b) The outcome that is best for the two prisoners as a group (neither confesses) is not what individually rational prisoners choose
- (c) If both prisoners act rationally, they will both choose not to confess
- (d) The game shows that people are irrational

**Question 94.** Six people each receive \$5 and can contribute any amount to a public pot. The pot is doubled and divided evenly among all six. Three people contribute their full \$5 and three contribute nothing. How much money does a person who contributed \$5 end up with, and how much does a person who contributed nothing end up with?

- (a) The contributors end with \$5 and the non-contributors end with \$10
- (b) The contributors end with \$10 and the non-contributors end with \$5
- (c) Everyone ends with \$10
- (d) The contributors end with \$15 and the non-contributors end with \$0

**Question 95.** Willer (2009) proposes that people contribute to collective action even though it is not in their short-term material interest because groups reward contributors with higher social status. Which of the following are true?<br>(1) This theory is consistent with rational choice theory (people make the best choice for them given the information available).<br>(2) Under this theory, the contributions are not in the contributor's self-interest at all.

- (a) (1) only
- (b) (2) only
- (c) Both (1) and (2)
- (d) Neither (1) nor (2)

**Question 96.** A researcher theorizes that people who know others can see whether they vote are more likely to vote, then mails voters letters that reveal their voting records to neighbors and compares their turnout to a control group. Which of the following correctly labels the parts of the scientific process?

- (a) Theory: people care about what others think of whether they vote<br>Hypothesis: people sent letters revealing their voting records to neighbors will vote more often<br>Empirical test: the randomized mailer experiment
- (b) Theory: the randomized mailer experiment<br>Hypothesis: people care about what others think<br>Empirical test: voting records will be revealed
- (c) Theory: people sent letters will vote more often<br>Hypothesis: people care about what others think<br>Empirical test: the randomized mailer experiment
- (d) Theory: the randomized mailer experiment<br>Hypothesis: the randomized mailer experiment<br>Empirical test: people care about what others think